# Model Training: VibroPredict Variants

Train three model variants for ablation:
1. **VibroPredict-Full**: All three branches (seq + spec + chem)
2. **VibroPredict-NoSpec**: Spectral branch dropped (sequence + chemistry only)
3. **VibroPredict-NoSeq**: Sequence branch zeroed (spectral + chemistry only)

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
from torch.utils.data import DataLoader

from vibropredict.data.enzyme_kinetics_dataset import EnzymeKineticsDataset
from vibropredict.models.vibropredict_hybrid import VibroPredictHybrid
from vibropredict.training.trainer import TrainerWithMMDrop
from vibropredict.training.losses import MutantRankingLoss
from vibropredict.training.metrics import compute_all_metrics

## Load Data

In [ ]:
train_dataset = EnzymeKineticsDataset(
    csv_path='../data/vibropredict/splits/train.csv',
    vdos_dir='../data/vibropredict/vdos/',
)
val_dataset = EnzymeKineticsDataset(
    csv_path='../data/vibropredict/splits/val.csv',
    vdos_dir='../data/vibropredict/vdos/',
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

## Train Full Model (with MM-Drop)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = VibroPredictHybrid(fusion_dim=512, dropout=0.2)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
loss_fn = MutantRankingLoss(lambda_rank=0.1)

trainer = TrainerWithMMDrop(
    model=model,
    optimizer=optimizer,
    device=device,
    checkpoint_dir='../checkpoints/vibropredict_full',
)

best_loss = trainer.fit(
    train_loader, val_loader, loss_fn,
    epochs=50, p_drop=0.25, patience=10,
)
print(f'Best validation loss: {best_loss:.4f}')

## Evaluate

In [ ]:
test_dataset = EnzymeKineticsDataset(
    csv_path='../data/vibropredict/splits/test.csv',
    vdos_dir='../data/vibropredict/vdos/',
)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

val_results = trainer.validate(test_loader, loss_fn)
metrics = compute_all_metrics(val_results['predictions'], val_results['targets'])

print('Test Metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')